# Conjunto de datos completo sin clusterización

In [1]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion_CTNET.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,77,0,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,82,0,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,85,0,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,87,0,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,88,0,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,86,0,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,89,0,6,Soleado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,95,0,7,Soleado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,100,0,8,Soleado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,100,1,9,Soleado,Lluvioso


In [2]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [3]:
datos_dia = datos[datos["Cluster GMM"] == "Soleado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
60,2022-09-03 12:00:00,17723.695569,21,58,4,12,Nublado,Soleado,11246.659307,27523.885172
62,2022-09-03 14:00:00,21548.984244,24,42,6,14,Nublado,Soleado,20400.000000,28500.000000
63,2022-09-03 15:00:00,25500.000000,24,39,7,15,Nublado,Soleado,21548.984244,24647.568577
64,2022-09-03 16:00:00,23122.803757,25,41,6,16,Nublado,Soleado,25500.000000,25500.000000
84,2022-09-04 12:00:00,22850.891263,21,66,6,12,Nublado,Soleado,17646.593401,17723.695569
85,2022-09-04 13:00:00,27000.000000,22,58,10,13,Nublado,Soleado,22850.891263,20400.000000
86,2022-09-04 14:00:00,25168.164594,23,51,12,14,Nublado,Soleado,27000.000000,21548.984244
87,2022-09-04 15:00:00,24300.000000,23,47,10,15,Nublado,Soleado,25168.164594,25500.000000
88,2022-09-04 16:00:00,19686.387416,24,46,7,16,Nublado,Soleado,24300.000000,23122.803757
109,2022-09-05 13:00:00,30000.000000,21,60,6,13,Nublado,Soleado,25558.205239,27000.000000


In [4]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [5]:
X = datos_dia[columns]
X

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
60,21,58,4,12,11246.659307,27523.885172
62,24,42,6,14,20400.000000,28500.000000
63,24,39,7,15,21548.984244,24647.568577
64,25,41,6,16,25500.000000,25500.000000
84,21,66,6,12,17646.593401,17723.695569
...,...,...,...,...,...,...
18254,23,46,9,13,26286.000000,25559.000000
18255,25,39,8,14,25653.000000,25579.000000
18277,21,51,6,12,26323.000000,26286.000000
18278,23,44,7,13,26277.000000,25653.000000


In [6]:
y = datos_dia[['Generación']]
y

,Generación
60,17723.695569
62,21548.984244
63,25500.000000
64,23122.803757
84,22850.891263
...,...
18254,25653.000000
18255,25362.000000
18277,26277.000000
18278,25598.000000


Dividimos entrenamiento, validación y prueba

In [7]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [8]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 1388, y_train: 1388
X_val: 298, y_val: 298
X_test: 298, y_test: 298


## Escalar con MinMaxScaler

In [9]:
from sklearn.preprocessing import MinMaxScaler

In [10]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [11]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.2        0.79104478 0.         0.28571429 0.37488864 0.91746284]
 [0.35       0.55223881 0.2        0.57142857 0.68       0.95      ]
 [0.35       0.50746269 0.3        0.71428571 0.71829947 0.82158562]
 ...
 [0.35       0.79104478 0.1        0.         0.8229     0.91186667]
 [0.5        0.58208955 0.5        0.14285714 0.90826667 0.9442    ]
 [0.65       0.40298507 0.8        0.28571429 0.94696667 0.93683333]]
(1388, 6)


In [12]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
60,0.20,0.791045,0.0,0.285714,0.374889,0.917463
62,0.35,0.552239,0.2,0.571429,0.680000,0.950000
63,0.35,0.507463,0.3,0.714286,0.718299,0.821586
64,0.40,0.537313,0.2,0.857143,0.850000,0.850000
84,0.20,0.910448,0.2,0.285714,0.588220,0.590790
...,...,...,...,...,...,...
15015,0.70,0.402985,0.8,0.571429,0.928933,0.932600
15016,0.75,0.328358,0.5,0.714286,0.932600,0.965300
15035,0.35,0.791045,0.1,0.000000,0.822900,0.911867
15036,0.50,0.582090,0.5,0.142857,0.908267,0.944200


In [13]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.75       0.29850746 1.         0.42857143 0.9369     0.92893333]
 [0.8        0.25373134 0.8        0.57142857 0.92893333 0.9326    ]
 [0.9        0.20895522 0.5        0.71428571 0.9326     0.97063333]
 ...
 [0.45       0.6119403  0.5        0.71428571 0.95016667 0.8617    ]
 [0.2        0.94029851 0.4        0.14285714 0.89683333 0.92476667]
 [0.3        0.82089552 0.8        0.28571429 0.92476667 0.93596667]]
(298, 6)


In [14]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
15038,0.75,0.298507,1.0,0.428571,0.936900,0.928933
15039,0.80,0.253731,0.8,0.571429,0.928933,0.932600
15040,0.90,0.208955,0.5,0.714286,0.932600,0.970633
15059,0.40,0.701493,0.1,0.000000,0.693167,0.908267
15060,0.55,0.507463,0.5,0.142857,0.727967,0.946967
...,...,...,...,...,...,...
16526,0.35,0.716418,1.0,0.428571,0.935967,0.848067
16527,0.45,0.641791,0.8,0.571429,0.943800,0.851767
16528,0.45,0.611940,0.5,0.714286,0.950167,0.861700
16548,0.20,0.940299,0.4,0.142857,0.896833,0.924767


In [15]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.35       0.73134328 0.8        0.42857143 0.9375     0.9438    ]
 [0.45       0.67164179 0.8        0.57142857 0.95086667 0.95016667]
 [0.45       0.64179104 0.5        0.71428571 0.95153333 0.95743333]
 ...
 [0.2        0.68656716 0.2        0.28571429 0.87743333 0.8762    ]
 [0.3        0.58208955 0.3        0.42857143 0.8759     0.8551    ]
 [0.35       0.49253731 0.4        0.57142857 0.85326667 0.8454    ]]
(298, 6)


In [16]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
16550,0.35,0.731343,0.8,0.428571,0.937500,0.943800
16551,0.45,0.671642,0.8,0.571429,0.950867,0.950167
16552,0.45,0.641791,0.5,0.714286,0.951533,0.957433
16572,0.25,0.895522,0.2,0.142857,0.933400,0.924767
16573,0.30,0.791045,0.8,0.285714,0.852400,0.937500
...,...,...,...,...,...,...
18254,0.30,0.611940,0.5,0.428571,0.876200,0.851967
18255,0.40,0.507463,0.4,0.571429,0.855100,0.852633
18277,0.20,0.686567,0.2,0.285714,0.877433,0.876200
18278,0.30,0.582090,0.3,0.428571,0.875900,0.855100


In [17]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [18]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.19047619 0.76811594 0.         0.28571429 0.37488864 0.91746284]
 [0.33333333 0.53623188 0.2        0.57142857 0.68       0.95      ]
 [0.33333333 0.49275362 0.3        0.71428571 0.71829947 0.82158562]
 ...
 [0.19047619 0.66666667 0.2        0.28571429 0.87743333 0.8762    ]
 [0.28571429 0.56521739 0.3        0.42857143 0.8759     0.8551    ]
 [0.33333333 0.47826087 0.4        0.57142857 0.85326667 0.8454    ]]
(1984, 6)


In [19]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
60,0.190476,0.768116,0.0,0.285714,0.374889,0.917463
62,0.333333,0.536232,0.2,0.571429,0.680000,0.950000
63,0.333333,0.492754,0.3,0.714286,0.718299,0.821586
64,0.380952,0.521739,0.2,0.857143,0.850000,0.850000
84,0.190476,0.884058,0.2,0.285714,0.588220,0.590790
...,...,...,...,...,...,...
18254,0.285714,0.594203,0.5,0.428571,0.876200,0.851967
18255,0.380952,0.492754,0.4,0.571429,0.855100,0.852633
18277,0.190476,0.666667,0.2,0.285714,0.877433,0.876200
18278,0.285714,0.565217,0.3,0.428571,0.875900,0.855100


In [20]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [21]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.59078985]
 [0.71829947]
 [0.85      ]
 ...
 [0.90826667]
 [0.94696667]
 [0.9369    ]]
(1388, 1)


In [22]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
60,0.590790
62,0.718299
63,0.850000
64,0.770760
84,0.761696
...,...
15015,0.932600
15016,0.970633
15035,0.908267
15036,0.946967


In [23]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.92893333]
 [0.9326    ]
 [0.97063333]
 [0.72796667]
 [0.751     ]
 [0.9429    ]
 [0.93506667]
 [0.885     ]
 [0.86473333]
 [0.8983    ]
 [0.95563333]
 [0.98136667]
 [0.97883333]
 [0.94683333]
 [0.92646667]
 [0.9025    ]
 [0.9426    ]
 [0.93683333]
 [0.93753333]
 [0.9363    ]
 [0.92646667]
 [0.90996667]
 [0.9332    ]
 [0.94056667]
 [0.93186667]
 [0.95676667]
 [0.83383333]
 [0.9332    ]
 [0.9537    ]
 [0.85843333]
 [0.77453333]
 [0.76263333]
 [0.9332    ]
 [0.9479    ]
 [0.95776667]
 [0.96316667]
 [0.9624    ]
 [0.9332    ]
 [0.9484    ]
 [0.96203333]
 [0.96876667]
 [0.95776667]
 [0.93553333]
 [0.94413333]
 [0.84376667]
 [0.84483333]
 [0.96416667]
 [0.94396667]
 [0.97616667]
 [0.97016667]
 [0.97313333]
 [0.94583333]
 [0.9501    ]
 [0.93826667]
 [0.9582    ]
 [0.95276667]
 [0.94963333]
 [0.95856667]
 [0.9593    ]
 [0.94006667]
 [0.96873333]
 [0.95253333]
 [0.95486667]
 [0.97816667]
 [0.95946667]
 [0.91103333]
 [0.94243333]
 [0.94176667]
 [0.93916667]
 [0.93993333]
 [0.9288    ]
 [0.72

In [24]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
15038,0.928933
15039,0.932600
15040,0.970633
15059,0.727967
15060,0.751000
...,...
16526,0.943800
16527,0.950167
16528,0.957433
16548,0.924767


In [25]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.95086667]
 [0.95153333]
 [0.95743333]
 [0.8524    ]
 [0.8321    ]
 [0.9522    ]
 [0.6613    ]
 [0.65823333]
 [0.39263333]
 [0.39453333]
 [0.46656667]
 [0.4486    ]
 [0.51253333]
 [0.66073333]
 [0.66416667]
 [0.688     ]
 [0.66283333]
 [0.92123333]
 [0.9215    ]
 [0.91553333]
 [0.91133333]
 [0.8989    ]
 [0.92446667]
 [0.92853333]
 [0.91553333]
 [0.91063333]
 [0.7407    ]
 [0.737     ]
 [0.83793333]
 [0.824     ]
 [0.82563333]
 [0.92456667]
 [0.94646667]
 [0.95036667]
 [0.94076667]
 [0.92476667]
 [0.92123333]
 [0.91896667]
 [0.91553333]
 [0.81956667]
 [0.92196667]
 [0.92623333]
 [0.92043333]
 [0.9242    ]
 [0.9156    ]
 [0.92236667]
 [0.8345    ]
 [0.72543333]
 [0.82626667]
 [0.9101    ]
 [0.92196667]
 [0.92103333]
 [0.9068    ]
 [0.91263333]
 [0.90873333]
 [0.92196667]
 [0.92593333]
 [0.90816667]
 [0.92106667]
 [0.9109    ]
 [0.92196667]
 [0.92456667]
 [0.90863333]
 [0.9157    ]
 [0.91783333]
 [0.92613333]
 [0.92293333]
 [0.9093    ]
 [0.90856667]
 [0.90463333]
 [0.8344    ]
 [0.92

In [26]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
16550,0.950867
16551,0.951533
16552,0.957433
16572,0.852400
16573,0.832100
...,...
18254,0.855100
18255,0.845400
18277,0.875900
18278,0.853267


In [27]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [28]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.59078985]
 [0.71829947]
 [0.85      ]
 ...
 [0.8759    ]
 [0.85326667]
 [0.84663333]]
(1984, 1)


In [29]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
60,0.590790
62,0.718299
63,0.850000
64,0.770760
84,0.761696
...,...
18254,0.855100
18255,0.845400
18277,0.875900
18278,0.853267


## Definición de modelos

### RandomForest

In [30]:
from lightgbm import LGBMRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score

In [31]:
# Inicializar listas para métricas
LightGBM_model = LGBMRegressor(num_leaves=500, subsample= 0.10698460631792395, colsample_bytree= 0.7272836809565294, min_data_in_leaf= 85)
LightGBM_model.fit(X_train_scaled_df, y_train_scaled_df)
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["LightGBM"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = LightGBM_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.255558 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 614
[LightGBM] [Info] Number of data points in the train set: 1388, number of used features: 6
[LightGBM] [Info] Start training from score 0.884575
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

In [32]:
resultados

,LightGBM
16550,28000.79621
16551,27441.039487
16552,27653.008343
16572,28695.531858
16573,26100.23055
...,...
18254,25624.012047
18255,25407.643926
18277,26526.444089
18278,26062.873849


In [33]:
predicciones = y_test.copy()
predicciones

,Generación
16550,28526.0
16551,28546.0
16552,28723.0
16572,25572.0
16573,24963.0
...,...
18254,25653.0
18255,25362.0
18277,26277.0
18278,25598.0


In [34]:
predicciones["LightGBM"] = resultados["LightGBM"]
predicciones

,Generación,LightGBM
16550,28526.0,28000.79621
16551,28546.0,27441.039487
16552,28723.0,27653.008343
16572,25572.0,28695.531858
16573,24963.0,26100.23055
...,...,...
18254,25653.0,25624.012047
18255,25362.0,25407.643926
18277,26277.0,26526.444089
18278,25598.0,26062.873849


In [35]:
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")

MAE: 1654.2360
RMSE: 3114.7331
R²: 0.6976


## Random Forest

In [36]:
from sklearn.ensemble import RandomForestRegressor

In [37]:
#Modelo LightGBM
RF_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=0,
    n_estimators=400,
    min_impurity_decrease=0,
    max_depth=None,
    bootstrap=True
)
RF_model.fit(X_train_scaled_df, y_train_scaled_df)
# Inicializar listas para métricas
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["Random Forest"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = RF_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [38]:
predicciones["Random Forest"] = resultados["Random Forest"]
predicciones

,Generación,LightGBM,Random Forest
16550,28526.0,28000.79621,28084.034609
16551,28546.0,27441.039487,27743.33194
16552,28723.0,27653.008343,28106.194443
16572,25572.0,28695.531858,28406.4
16573,24963.0,26100.23055,26665.619348
...,...,...,...
18254,25653.0,25624.012047,26388.864157
18255,25362.0,25407.643926,25765.293528
18277,26277.0,26526.444089,26570.909285
18278,25598.0,26062.873849,26251.964527


## Preparación redes neuronales

In [39]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [40]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [41]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (1340, 48, 6), y_train: (1340, 1)
X_val: (250, 48, 6), y_val: (250, 1)
X_test: (250, 48, 6), y_test: (250, 1)


## CTNET

In [42]:
import tensorflow as tf
from tensorflow.keras import layers

In [43]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    
    for _ in range(num_transformer_blocks):
        enc_out = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(enc_out, enc_out)
    res = x + enc_out
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    x = layers.Dense(832, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(mlp_dropout)(x)

    outputs = layers.Dense(1)(x)
    
    return tf.keras.Model(inputs, outputs)

In [44]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=4, num_heads=3, ff_dim=32, num_transformer_blocks=3, mlp_units=[256], mlp_dropout=0.3, dropout=0.2)

In [45]:
history = compile_and_fit(CTNET)

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 68s 4s/step - loss: 0.8183 - mean_absolute_error: 0.8944 - mean_absolute_percentage_error: 2147.2354 - root_mean_squared_error: 0.9046 - val_loss: 0.7526 - val_mean_absolute_error: 0.8611 - val_mean_absolute_percentage_error: 99.5525 - val_root_mean_squared_error: 0.8675
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 11s 4s/step - loss: 0.8091 - mean_absolute_error: 0.8890 - mean_absolute_percentage_error: 35967.1016 - root_mean_squared_error: 0.8995 - val_loss: 0.7453 - val_mean_absolute_error: 0.8568 - val_mean_absolute_percentage_error: 99.0519 - val_root_mean_squared_error: 0.8633
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - loss: 0.8046 - mean_absolute_error: 0.8879 - mean_absolute_percentage_error: 67292.2656 - root_mean_squared_error: 0.8970 - val_loss: 0.7373 - val_mean_absolute_error: 0.8522 - val_mean_absolute_percentage_error: 98.5018 - val_root_mean_squared_error: 0.8587
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 4s/step - loss: 0.7924 - mean_absol

In [46]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

8/8 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step


array([[0.87212104],
       [0.871561  ],
       [0.8701927 ],
       [0.87201685],
       [0.8719718 ],
       [0.8712695 ],
       [0.8704179 ],
       [0.8702238 ],
       [0.87077135],
       [0.8721224 ],
       [0.87349445],
       [0.8735046 ],
       [0.87489945],
       [0.87598854],
       [0.87706894],
       [0.87765074],
       [0.8790534 ],
       [0.88020366],
       [0.8797138 ],
       [0.88020486],
       [0.88059384],
       [0.8811379 ],
       [0.88207364],
       [0.8816673 ],
       [0.8815842 ],
       [0.8828508 ],
       [0.8829713 ],
       [0.8828495 ],
       [0.8828546 ],
       [0.8825051 ],
       [0.88320875],
       [0.8824282 ],
       [0.88301367],
       [0.88239974],
       [0.88306314],
       [0.8852775 ],
       [0.88347125],
       [0.883336  ],
       [0.8825051 ],
       [0.8819286 ],
       [0.88242894],
       [0.88073325],
       [0.88029104],
       [0.87971294],
       [0.879996  ],
       [0.88122743],
       [0.8800948 ],
       [0.879

In [47]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [48]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [49]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
16550,28526.0,28000.79621,28084.034609,NaN
16551,28546.0,27441.039487,27743.33194,NaN
16552,28723.0,27653.008343,28106.194443,NaN
16572,25572.0,28695.531858,28406.4,NaN
16573,24963.0,26100.23055,26665.619348,NaN
...,...,...,...,...
18254,25653.0,25624.012047,26388.864157,25714.863281
18255,25362.0,25407.643926,25765.293528,25758.392578
18277,26277.0,26526.444089,26570.909285,25746.933594
18278,25598.0,26062.873849,26251.964527,25747.503906


In [50]:
predicciones["CTNET"] = predicciones["CTNET"].fillna(0)

In [51]:
# import optuna
# import tensorflow as tf
# from tensorflow.keras import layers
# from sklearn.model_selection import train_test_split

# # Definir la función objetivo para Optuna
# def objective(trial):
#     # Sugerir valores para los hiperparámetros
#     head_size = trial.suggest_int("head_size", 8, 64, step=8)
#     num_heads = trial.suggest_int("num_heads", 2, 8, step=2)
#     ff_dim = trial.suggest_int("ff_dim", 32, 256, step=32)
#     num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 4)
#     mlp_units = trial.suggest_categorical("mlp_units", [[128, 64], [256, 128, 64], [512, 256, 128]])
#     dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
#     mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5, step=0.1)
#     learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)

#     # Construcción del modelo con los hiperparámetros sugeridos
#     model = build_model(
#         input_shape=X_train_windowed.shape[1:],
#         head_size=head_size,
#         num_heads=num_heads,
#         ff_dim=ff_dim,
#         num_transformer_blocks=num_transformer_blocks,
#         mlp_units=mlp_units,
#         dropout=dropout,
#         mlp_dropout=mlp_dropout
#     )

#     # Compilar el modelo con los hiperparámetros sugeridos
#     model.compile(
#         loss=tf.keras.losses.MeanSquaredError(),
#         optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()]
#     )

#     # Entrenamiento con un número reducido de épocas para acelerar la búsqueda
#     history = model.fit(
#         X_train_windowed, y_train_windowed,
#         validation_split=0.2,
#         epochs=50,  # Reducimos las épocas para acelerar la búsqueda
#         batch_size=512,
#         verbose=0
#     )

#     # Obtener la métrica de validación (RMSE) y minimizarla
#     val_rmse = min(history.history["val_root_mean_squared_error"])
    
#     return val_rmse  # Queremos minimizar el RMSE

# # Ejecutar la optimización de hiperparámetros
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=20, timeout=3600)  # 20 iteraciones, máximo 1 hora

# # Mostrar los mejores hiperparámetros encontrados
# best_params = study.best_params
# print(f"Mejores hiperparámetros: {best_params}")


In [52]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=16, num_heads=8, ff_dim=256, num_transformer_blocks=1, mlp_units=[128,64], mlp_dropout=0.2, dropout=0.2)

In [53]:
history = compile_and_fit(CTNET, learning_rate = 0.0010762230908145116)

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 145s 9s/step - loss: 0.8069 - mean_absolute_error: 0.8878 - mean_absolute_percentage_error: 41334.4766 - root_mean_squared_error: 0.8983 - val_loss: 0.6770 - val_mean_absolute_error: 0.8160 - val_mean_absolute_percentage_error: 94.2462 - val_root_mean_squared_error: 0.8228
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - loss: 0.7168 - mean_absolute_error: 0.8368 - mean_absolute_percentage_error: 490076.4062 - root_mean_squared_error: 0.8466 - val_loss: 0.5334 - val_mean_absolute_error: 0.7227 - val_mean_absolute_percentage_error: 83.2632 - val_root_mean_squared_error: 0.7304
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 21s 7s/step - loss: 0.5560 - mean_absolute_error: 0.7355 - mean_absolute_percentage_error: 1415517.2500 - root_mean_squared_error: 0.7455 - val_loss: 0.3067 - val_mean_absolute_error: 0.5437 - val_mean_absolute_percentage_error: 62.1860 - val_root_mean_squared_error: 0.5538
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 19s 4s/step - loss: 0.3083 - mean

In [54]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

8/8 ━━━━━━━━━━━━━━━━━━━━ 15s 704ms/step


array([[0.8726911 ],
       [0.87249213],
       [0.8689657 ],
       [0.869546  ],
       [0.87474954],
       [0.8749793 ],
       [0.87496156],
       [0.8757536 ],
       [0.8747025 ],
       [0.8798923 ],
       [0.8827706 ],
       [0.8851206 ],
       [0.88753414],
       [0.8864283 ],
       [0.88982433],
       [0.89180475],
       [0.8937475 ],
       [0.8955782 ],
       [0.8932132 ],
       [0.8933301 ],
       [0.89558744],
       [0.89734334],
       [0.89707744],
       [0.89603144],
       [0.89731175],
       [0.8987145 ],
       [0.9001654 ],
       [0.89907223],
       [0.89972466],
       [0.8998817 ],
       [0.9002709 ],
       [0.90158635],
       [0.9002516 ],
       [0.9000081 ],
       [0.9021053 ],
       [0.90312815],
       [0.90267366],
       [0.9016976 ],
       [0.9010266 ],
       [0.8992475 ],
       [0.90041167],
       [0.8989498 ],
       [0.8975012 ],
       [0.8979803 ],
       [0.8978551 ],
       [0.89889115],
       [0.89881223],
       [0.897

In [55]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [56]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [57]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
16550,28526.0,28000.79621,28084.034609,NaN
16551,28546.0,27441.039487,27743.33194,NaN
16552,28723.0,27653.008343,28106.194443,NaN
16572,25572.0,28695.531858,28406.4,NaN
16573,24963.0,26100.23055,26665.619348,NaN
...,...,...,...,...
18254,25653.0,25624.012047,26388.864157,23417.914062
18255,25362.0,25407.643926,25765.293528,23496.564453
18277,26277.0,26526.444089,26570.909285,23499.189453
18278,25598.0,26062.873849,26251.964527,23467.341797


## Forecasting

In [58]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping

In [59]:
Forecast_model = Sequential()
Forecast_model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

#CNN
Forecast_model.add(Conv1D(filters=64, kernel_size=2, padding='same', activation='relu'))
Forecast_model.add(BatchNormalization())  # 🔹 Nueva Normalización aquí
Forecast_model.add(MaxPooling1D(pool_size=2))

#model_Soleado.add(Flatten())
#BiLSTM
Forecast_model.add(Bidirectional(LSTM(128, return_sequences=True)))
Forecast_model.add(Bidirectional(LSTM(64, return_sequences=True)))
Forecast_model.add(Dropout(0.2))  # 🔹 Mayor regularización en BiLSTM
Forecast_model.add(Bidirectional(LSTM(32, return_sequences=False)))

#Normalización y Dropout
Forecast_model.add(BatchNormalization())
Forecast_model.add(Dropout(0.3))

# Capas Densas
Forecast_model.add(Dense(16, activation='relu'))
Forecast_model.add(Dense(1, 'relu'))

Forecast_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 48, 64)         │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 24, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,601 (1.55 MB)

 Trainable params: 405,345 (1.55 MB)

 Non-trainable params: 256 (1.00 KB)

In [60]:
cp = ModelCheckpoint('Forcasting_model.keras', save_best_only=True)
Forecast_model.compile(optimizer=Adam(learning_rate=0.0001), loss=Huber(delta=1000), metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [61]:
history = Forecast_model.fit(X_train_windowed, y_train_windowed, validation_data=(X_val_windowed, y_val_windowed), epochs=100, batch_size=8, callbacks=[cp, early_stop])

Epoch 1/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 328s 487ms/step - loss: 0.2298 - mae: 0.5852 - val_loss: 0.2573 - val_mae: 0.6950
Epoch 2/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 89s 477ms/step - loss: 0.1855 - mae: 0.5148 - val_loss: 0.1726 - val_mae: 0.5542
Epoch 3/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 84s 465ms/step - loss: 0.1635 - mae: 0.4744 - val_loss: 0.1311 - val_mae: 0.4674
Epoch 4/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 72s 386ms/step - loss: 0.1581 - mae: 0.4579 - val_loss: 0.0858 - val_mae: 0.3270
Epoch 5/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 96s 444ms/step - loss: 0.1316 - mae: 0.4080 - val_loss: 0.0663 - val_mae: 0.2991
Epoch 6/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 83s 420ms/step - loss: 0.1121 - mae: 0.3778 - val_loss: 0.0594 - val_mae: 0.2728
Epoch 7/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 85s 404ms/step - loss: 0.1109 - mae: 0.3631 - val_loss: 0.0707 - val_mae: 0.2923
Epoch 8/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 94s 447ms/step - loss: 0.1094 - mae: 0.3618 - val_loss: 0.0559 - val_mae: 0.2477
Epoch 9/100
168/168 ━━━

In [62]:
Forecast_predictions = Forecast_model.predict(X_test_windowed)
Forecast_predictions

8/8 ━━━━━━━━━━━━━━━━━━━━ 65s 4s/step


array([[0.9218264 ],
       [0.8446328 ],
       [1.0652634 ],
       [1.0239036 ],
       [1.0946991 ],
       [1.0523978 ],
       [1.099423  ],
       [1.1439434 ],
       [1.1041492 ],
       [1.0478956 ],
       [0.9613395 ],
       [0.8591307 ],
       [0.78919536],
       [0.7996155 ],
       [0.75218964],
       [0.6940972 ],
       [0.6398144 ],
       [0.62347853],
       [0.6627694 ],
       [0.6594491 ],
       [0.6360996 ],
       [0.6954222 ],
       [0.7163104 ],
       [0.76867485],
       [0.720865  ],
       [0.71936387],
       [0.73646075],
       [0.7602855 ],
       [0.7419674 ],
       [0.68399656],
       [0.68718386],
       [0.6713588 ],
       [0.68194354],
       [0.6701453 ],
       [0.6287475 ],
       [0.6519225 ],
       [0.67223   ],
       [0.7283652 ],
       [0.717574  ],
       [0.73067856],
       [0.75270617],
       [0.7437881 ],
       [0.7496493 ],
       [0.7207644 ],
       [0.667727  ],
       [0.6848287 ],
       [0.6731316 ],
       [0.706

In [63]:
Forecast_predictions = y_scaler.inverse_transform(Forecast_predictions.reshape(-1, 1))
Forecast_predictions = np.clip(Forecast_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [64]:
Forecast_resultados = pd.DataFrame(Forecast_predictions, index = y_test_windowed.index, columns=["Forecast"])

In [65]:
predicciones["Forecast"] = Forecast_resultados["Forecast"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast
16550,28526.0,28000.79621,28084.034609,NaN,NaN
16551,28546.0,27441.039487,27743.33194,NaN,NaN
16552,28723.0,27653.008343,28106.194443,NaN,NaN
16572,25572.0,28695.531858,28406.4,NaN,NaN
16573,24963.0,26100.23055,26665.619348,NaN,NaN
...,...,...,...,...,...
18254,25653.0,25624.012047,26388.864157,23417.914062,15013.654297
18255,25362.0,25407.643926,25765.293528,23496.564453,13993.070312
18277,26277.0,26526.444089,26570.909285,23499.189453,16085.019531
18278,25598.0,26062.873849,26251.964527,23467.341797,6692.576172


## Métricas

In [66]:
predicciones.loc[~predicciones['CTNET'].isna(),'Generación']

16815    24788.0
16816    27303.0
16836    27659.0
16837    27631.0
16838    27204.0
          ...   
18254    25653.0
18255    25362.0
18277    26277.0
18278    25598.0
18279    25399.0
Name: Generación, Length: 250, dtype: float64

## Photovoltaic

In [67]:
from tensorflow.keras.models import Model
inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

# Primera capa CNN
x = Conv1D(filters=64, kernel_size=4, padding='same', activation='relu')(inputs)
x = MaxPooling1D(pool_size=2)(x)

# Segunda capa CNN
x = Conv1D(filters=128, kernel_size=4, padding='same', activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)

# Capa BiGRU
x = Bidirectional(GRU(64, return_sequences=True))(x)

# Atención: se define de forma explícita
attention = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)

# Aplanar y agregar Dropout
x = Flatten()(attention)
x = Dropout(0.4)(x)
initializer = tf.keras.initializers.HeNormal()
x = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x)
x = Dense(32, activation="relu")(x)  # Otra capa intermedia

# Capa de salida
outputs = Dense(1, activation="linear")(x)

# Definir el modelo
Photo_model = Model(inputs=inputs, outputs=outputs)

# Resumen del modelo
Photo_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 48, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 48, 64)    │      1,600 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 24, 64)    │          0 │ conv1d_13[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 24, 128)   │     32,896 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 12, 128)   │          0 │ conv1d_14[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 12, 128)   │     74,496 │ max_pooling1d_2[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 12, 128)   │    263,808 │ bidirectional_3[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1536)      │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 1536)      │          0 │ flatten[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │     98,368 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      2,080 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         33 │ dense_11[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 473,281 (1.81 MB)

 Trainable params: 473,281 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
cp2 = ModelCheckpoint('Photovoltaic_model.keras', save_best_only=True)
Photo_model.compile(optimizer=Adam(learning_rate=0.0001), loss="mean_squared_error", metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [69]:
history = Photo_model.fit(
    X_train_windowed, y_train_windowed,
    validation_data=(X_val_windowed, y_val_windowed),
    epochs=50,
    batch_size=16,
    callbacks=[cp, early_stop]
)

Epoch 1/50


84/84 ━━━━━━━━━━━━━━━━━━━━ 205s 556ms/step - loss: 1.4312 - mae: 0.4158 - val_loss: 0.9585 - val_mae: 0.0966
Epoch 2/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 64s 297ms/step - loss: 0.8721 - mae: 0.0874 - val_loss: 0.7043 - val_mae: 0.0964
Epoch 3/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 49s 318ms/step - loss: 0.6365 - mae: 0.0894 - val_loss: 0.5124 - val_mae: 0.1040
Epoch 4/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 42s 392ms/step - loss: 0.4606 - mae: 0.0874 - val_loss: 0.3735 - val_mae: 0.0988
Epoch 5/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 35s 320ms/step - loss: 0.3341 - mae: 0.0883 - val_loss: 0.2773 - val_mae: 0.0944
Epoch 6/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 38s 217ms/step - loss: 0.2402 - mae: 0.0844 - val_loss: 0.2042 - val_mae: 0.1118
Epoch 7/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 34s 336ms/step - loss: 0.1792 - mae: 0.0870 - val_loss: 0.1555 - val_mae: 0.1060
Epoch 8/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 44s 397ms/step - loss: 0.1309 - mae: 0.0826 - val_loss: 0.1267 - val_mae: 0.0964
Epoch 9/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 45s 410ms/step - los

In [70]:
Photo_predictions = Photo_model.predict(X_test_windowed)
Photo_predictions

6/8 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step 

8/8 ━━━━━━━━━━━━━━━━━━━━ 36s 2s/step


array([[0.8821268 ],
       [0.813296  ],
       [0.86029994],
       [0.9017311 ],
       [0.8786804 ],
       [0.8796775 ],
       [0.89616364],
       [0.90947884],
       [0.920426  ],
       [0.9074693 ],
       [0.91090506],
       [0.8978666 ],
       [0.89998966],
       [0.917474  ],
       [0.90970993],
       [0.906845  ],
       [0.89216065],
       [0.89585215],
       [0.9157665 ],
       [0.9185402 ],
       [0.9136083 ],
       [0.9033811 ],
       [0.89861166],
       [0.90283614],
       [0.85737133],
       [0.86131465],
       [0.8821773 ],
       [0.8986905 ],
       [0.89702135],
       [0.8923531 ],
       [0.88754743],
       [0.87391615],
       [0.89705175],
       [0.86261433],
       [0.79543716],
       [0.7333737 ],
       [0.77214974],
       [0.85859686],
       [0.86413187],
       [0.8998439 ],
       [0.89792246],
       [0.8755396 ],
       [0.90761566],
       [0.91750926],
       [0.88636035],
       [0.8969802 ],
       [0.88157344],
       [0.910

In [71]:
Photo_predictions = y_scaler.inverse_transform(Photo_predictions.reshape(-1, 1))
Photo_predictions = np.clip(Photo_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [72]:
Photo_resultados = pd.DataFrame(Photo_predictions, index = y_test_windowed.index, columns=["Photo"])

In [73]:
predicciones["Photo"] = Photo_resultados["Photo"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast,Photo
16550,28526.0,28000.79621,28084.034609,NaN,NaN,NaN
16551,28546.0,27441.039487,27743.33194,NaN,NaN,NaN
16552,28723.0,27653.008343,28106.194443,NaN,NaN,NaN
16572,25572.0,28695.531858,28406.4,NaN,NaN,NaN
16573,24963.0,26100.23055,26665.619348,NaN,NaN,NaN
...,...,...,...,...,...,...
18254,25653.0,25624.012047,26388.864157,23417.914062,15013.654297,25079.843750
18255,25362.0,25407.643926,25765.293528,23496.564453,13993.070312,25517.880859
18277,26277.0,26526.444089,26570.909285,23499.189453,16085.019531,25040.558594
18278,25598.0,26062.873849,26251.964527,23467.341797,6692.576172,25872.761719


In [74]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['Random Forest'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")

LightGBM
MAE: 1654.2360
RMSE: 3114.7331
R²: 0.6976
Random Forest
MAE: 1717.0140
RMSE: 3069.3600
R²: 0.7063
CTNET
MAE: 4013.1179
RMSE: 6267.6130
R²: -0.1586
Forecast
MAE: 11959.2745
RMSE: 15365.9355
R²: -5.9638
Photovoltaic
MAE: 3065.4999
RMSE: 4982.9021
R²: 0.2677


In [75]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones.loc[:, "LightGBM":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo
24,2022-09-02 00:00:00,0.0,19,76,0,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,81,0,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,84,0,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,86,0,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,86,0,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## X_train para hacer análisis de sobreajuste

In [76]:
predicciones_train = y_train.copy()

In [77]:
LightGBM_predictions_train = LightGBM_model.predict(X_train_scaled_df)
LightGBM_predictions_train = y_scaler.inverse_transform(LightGBM_predictions_train.reshape(-1, 1))
LightGBM_predictions_train = np.clip(LightGBM_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
LightGBM_resultados = pd.DataFrame(LightGBM_predictions_train, index = y_train_scaled_df.index, columns=["LightGBM_train"])
predicciones_train["LightGBM_train"] = LightGBM_resultados["LightGBM_train"]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85


In [78]:
RandomForest_predictions_train = RF_model.predict(X_train_scaled_df)
RandomForest_predictions_train = y_scaler.inverse_transform(RandomForest_predictions_train.reshape(-1, 1))
RandomForest_predictions_train = np.clip(RandomForest_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
RandomForest_resultados = pd.DataFrame(RandomForest_predictions_train, index = y_train_scaled_df.index, columns=["RandomForest_train"])
predicciones_train["RandomForest_train"] = RandomForest_resultados["RandomForest_train"]

In [79]:
CTNET_predictions_train = CTNET.predict(X_train_windowed)
CTNET_predictions_train = y_scaler.inverse_transform(CTNET_predictions_train.reshape(-1, 1))
CTNET_predictions_train = np.clip(CTNET_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
CTNET_resultados = pd.DataFrame(CTNET_predictions_train, index = y_train_windowed.index, columns=["CTNET_train"])
predicciones_train["CTNET_train"] = CTNET_resultados["CTNET_train"]

42/42 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step


In [80]:
Forecast_predictions_train = Forecast_model.predict(X_train_windowed)
Forecast_predictions_train = y_scaler.inverse_transform(Forecast_predictions_train.reshape(-1, 1))
Forecast_predictions_train = np.clip(Forecast_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Forecast_resultados = pd.DataFrame(Forecast_predictions_train, index = y_train_windowed.index, columns=["Forecast_train"])
predicciones_train["Forecast_train"] = Forecast_resultados["Forecast_train"]

42/42 ━━━━━━━━━━━━━━━━━━━━ 12s 138ms/step


In [81]:
Photo_predictions_train = Photo_model.predict(X_train_windowed)
Photo_predictions_train = y_scaler.inverse_transform(Photo_predictions_train.reshape(-1, 1))
Photo_predictions_train = np.clip(Photo_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Photo_resultados = pd.DataFrame(Photo_predictions_train, index = y_train_windowed.index, columns=["Photo_train"])
predicciones_train["Photo_train"] = Photo_resultados["Photo_train"]

42/42 ━━━━━━━━━━━━━━━━━━━━ 12s 79ms/step


In [82]:
predicciones_train

,Generación,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
60,17723.695569,18408.915820,17545.731151,NaN,NaN,NaN
62,21548.984244,17569.041197,21494.093666,NaN,NaN,NaN
63,25500.000000,20654.095675,24452.052101,NaN,NaN,NaN
64,23122.803757,24667.524780,23947.572378,NaN,NaN,NaN
84,22850.891263,16658.483574,21862.076445,NaN,NaN,NaN
...,...,...,...,...,...,...
15015,27978.000000,27534.177708,27838.222500,27428.035156,25066.638672,27297.425781
15016,29119.000000,27768.920192,28719.702500,27427.982422,25372.681641,27150.839844
15035,27248.000000,26921.211591,27329.209666,27429.828125,25485.687500,27527.341797
15036,28409.000000,28519.728260,28334.940000,27410.865234,25117.064453,27150.839844


In [83]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['LightGBM_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['RandomForest_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")

LightGBM
MAE: 1126.8062
RMSE: 2270.1274
R²: 0.6779
Random Forest
MAE: 401.4045
RMSE: 776.9427
R²: 0.9623
CTNET
MAE: 2128.3299
RMSE: 3600.4636
R²: 0.1555
Forecast
MAE: 7040.2763
RMSE: 10030.2871
R²: -5.5541
Photovoltaic
MAE: 1601.5441
RMSE: 2637.8954
R²: 0.5467


In [84]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones_train.loc[:, "LightGBM_train":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
24,2022-09-02 00:00:00,0.0,19,76,0,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,81,0,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,84,0,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,86,0,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,86,0,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
datos.to_excel("04.10_Predicciones_Conjunto_soleado GMM CTNET.xlsx", index=True)

## Guardamos los modelos

In [86]:
import joblib

# Guardar modelo LightGBM
joblib.dump(LightGBM_model, "4_10_LightGBM_model.pkl")

# Guardar modelo Random Forest
joblib.dump(RF_model, "4_10_RandomForest_model.pkl")


['4_10_RandomForest_model.pkl']

In [87]:
CTNET.save("4_10_CTNET_model.keras")
Forecast_model.save("4_10_Forecast_model.keras")
Photo_model.save("4_10_Photo_model.keras")